 Imports

In [1]:
import mlflow
import pandas as pd
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import os

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("flickr8k-image-captioning")

<Experiment: artifact_location='file:///D:/Projects/internship-project/notebooks/mlruns/1', creation_time=1784194096153, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784194096153, lifecycle_stage='active', name='flickr8k-image-captioning', tags={}, trace_location=None, workspace='default'>

Log Day 2 Baseline Run Into MLflow

In [2]:
baseline_metrics = pd.read_csv("blip_baseline_evaluation_report.csv")
metrics_dict = dict(zip(baseline_metrics['Metric'], baseline_metrics['Average Score']))

with mlflow.start_run(run_name="blip_zeroshot_baseline"):
    mlflow.log_param("model", "Salesforce/blip-image-captioning-base")
    mlflow.log_param("mode", "zero-shot")
    mlflow.log_param("validation_size", 200)
    mlflow.log_param("fine_tuned", False)

    for metric_name, score in metrics_dict.items():
        if score != "N/A":
            mlflow.log_metric(metric_name.lower(), float(score))

    mlflow.log_artifact("blip_baseline_evaluation_report.csv")
    mlflow.log_artifact("blip_baseline_full_results.csv")

print("Baseline run logged to MLflow.")

Baseline run logged to MLflow.


Load Model & Prepare Small Training Subset

In [3]:
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

df = pd.read_csv("../data/captions.txt")
images_dir = "../data/Images"

train_size = 20
train_images = df['image'].unique()[:train_size]
train_df = df[df['image'].isin(train_images)].reset_index(drop=True)
print(f"Training subset: {len(train_images)} images, {len(train_df)} captions")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Training subset: 20 images, 100 captions


  Freeze Vision Encoder

In [4]:
for param in model.vision_model.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

Trainable parameters: 137,881,148 / 223,971,644


Dataset Class

In [5]:
from torch.utils.data import Dataset, DataLoader

class FlickrDataset(Dataset):
    def __init__(self, df, images_dir, processor):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.images_dir, row['image'])).convert('RGB')
        encoding = self.processor(images=image, text=row['caption'], padding="max_length", return_tensors="pt")
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        return encoding

train_dataset = FlickrDataset(train_df, images_dir, processor)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
print(f"Batches per epoch: {len(train_loader)}")

Batches per epoch: 25


 Fine-Tuning Loop

In [6]:
from torch.optim import AdamW

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

epoch_losses = []

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    pixel_values = batch["pixel_values"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    outputs = model(input_ids=input_ids, pixel_values=pixel_values,
                     attention_mask=attention_mask, labels=input_ids)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_losses.append(loss.item())

avg_loss = sum(epoch_losses) / len(epoch_losses)
print(f"Average training loss: {avg_loss:.4f}")

Average training loss: 7.8977
